In [1]:
import os
import mne
import numpy as np

def crop_ec_segment(raw, discard_sec=10):
    # Find BAD_ACQ_SKIP segments
    bad_skips = [
        (onset, onset + duration)
        for onset, duration, desc in zip(
            raw.annotations.onset,
            raw.annotations.duration,
            raw.annotations.description
        )
        if desc == "BAD_ACQ_SKIP"
    ]
    if len(bad_skips) < 2:
        raise ValueError("Not enough BAD_ACQ_SKIP markers to define EC segment.")

    # Use the second segment as EC and discard the first N seconds
    ec_start = bad_skips[1][1] + discard_sec
    raw_ec = raw.copy().crop(tmin=ec_start)

    return raw_ec


def bandpower_from_epochs(epochs, bands):
    # Compute PSD on epochs (more robust than single PSD on continuous raw)
    psd = epochs.compute_psd(fmin=1, fmax=40, method="welch")
    freqs = psd.freqs
    data = psd.get_data()  # shape: (n_epochs, n_channels, n_freqs)

    feats = {}
    for band_name, (fmin, fmax) in bands.items():
        idx = (freqs >= fmin) & (freqs <= fmax)
        # Mean power across epochs, channels, and freqs
        feats[f"{band_name}_power_global"] = data[:, :, idx].mean()
        # Channel-wise power (mean across epochs & freqs)
        feats[f"{band_name}_power_by_channel"] = data[:, :, idx].mean(axis=(0, 2))
    return feats


In [2]:
import numpy as np
import pandas as pd
import mne
import os

# ================================
# 1️⃣ Core Processing Function
# ================================

def process_one_subject(file_path, subject_id):

    result = {}
    qc = {}
    
    try:
        # ------------------------
        # LOAD
        # ------------------------
        raw = mne.io.read_raw_egi(file_path, preload=True, verbose=False)

        # ------------------------
        # CROP EC
        # ------------------------
        raw_ec = crop_ec_segment(raw, discard_sec=10)
        raw_ec.pick_channels([ch for ch in raw_ec.ch_names if ch.startswith('E')])

        # ------------------------
        # RESAMPLE + FILTER
        # ------------------------
        raw_ec.resample(250, verbose=False)
        raw_ec.notch_filter(60, verbose=False)
        raw_ec.filter(1.0, 40, verbose=False)

        # ------------------------
        # BAD CHANNEL DETECTION
        # ------------------------
        raw_tmp = raw_ec.copy().load_data()
        data_uv = raw_tmp.get_data() * 1e6
        ch_names = raw_tmp.ch_names

        stds = np.std(data_uv, axis=1)
        median_std = np.median(stds)

        flat_bad = []
        var_bad = []
        line_bad = []

        # variance criterion
        for i, ch in enumerate(ch_names):
            if stds[i] > 5 * median_std:
                var_bad.append(ch)

        # line noise
        psd_line = raw_tmp.compute_psd(fmin=55, fmax=65, verbose=False)
        psd_data = psd_line.get_data()
        freqs = psd_line.freqs
        idx_60 = np.argmin(np.abs(freqs - 60))
        line_power = psd_data[:, idx_60]
        median_line = np.median(line_power)
        sd_line = np.std(line_power)

        for i, ch in enumerate(ch_names):
            if line_power[i] > median_line + 4 * sd_line:
                line_bad.append(ch)

        final_bad = sorted(list(set(var_bad + line_bad)))

        qc['n_bad_channels'] = len(final_bad)

        raw_ec.info['bads'] = final_bad
        raw_ec.interpolate_bads(reset_bads=True)
        raw_ec.set_eeg_reference('average')

        # ------------------------
        # EPOCH
        # ------------------------
        epochs = mne.make_fixed_length_epochs(
            raw_ec, duration=2.0, overlap=1.0, preload=True, verbose=False
        )

        qc['n_epochs_total'] = len(epochs)

        epochs_clean = epochs.copy().drop_bad(reject=dict(eeg=200e-6))

        qc['n_epochs_kept'] = len(epochs_clean)

        if qc['n_epochs_kept'] < 100:
            qc['flag_low_epochs'] = True
        else:
            qc['flag_low_epochs'] = False

        # ------------------------
        # PSD
        # ------------------------
        psd = epochs_clean.compute_psd(fmin=1, fmax=40, verbose=False)
        freqs = psd.freqs
        psd_mean = psd.get_data().mean(axis=0)

        # ------------------------
        # ROI
        # ------------------------
        frontal_roi = ['E3','E6','E8','E9']
        posterior_roi = ['E26','E34','E45','E35','E37','E38']

        frontal_roi = [ch for ch in frontal_roi if ch in raw_ec.ch_names]
        posterior_roi = [ch for ch in posterior_roi if ch in raw_ec.ch_names]

        fr_idx = [raw_ec.ch_names.index(ch) for ch in frontal_roi]
        po_idx = [raw_ec.ch_names.index(ch) for ch in posterior_roi]
        gl_idx = list(range(len(raw_ec.ch_names)))

        def band_power(ch_idx, fmin, fmax):
            idx = (freqs >= fmin) & (freqs <= fmax)
            return np.log(psd_mean[ch_idx][:, idx].mean())

        # ------------------------
        # FEATURES
        # ------------------------
        result['posterior_alpha'] = band_power(po_idx, 8, 12)
        result['posterior_beta']  = band_power(po_idx, 13, 30)
        result['frontal_theta']   = band_power(fr_idx, 4, 7)

        theta = band_power(fr_idx, 4, 7)
        beta  = band_power(fr_idx, 13, 30)
        result['frontal_theta_beta_ratio'] = theta - beta

        result['global_alpha'] = band_power(gl_idx, 8, 12)
        result['global_beta']  = band_power(gl_idx, 13, 30)

        # IAF
        po_psd = psd_mean[po_idx].mean(axis=0)
        alpha_range = (freqs >= 7) & (freqs <= 13)
        result['IAF'] = freqs[alpha_range][np.argmax(po_psd[alpha_range])]

        # Aperiodic
        log_freqs = np.log10(freqs[1:])
        log_psd = np.log10(psd_mean.mean(axis=0)[1:])
        coef = np.polyfit(log_freqs, log_psd, 1)
        result['aperiodic_exponent'] = -coef[0]

        # ------------------------
        # QC Flags
        # ------------------------
        result.update(qc)
        result['Subject'] = subject_id

        result['qc_flag'] = (
            (qc['n_bad_channels'] > 15) or
            (qc['n_epochs_kept'] < 100)
        )

        return result

    except Exception as e:
        return {
            'Subject': subject_id,
            'processing_failed': True,
            'error': str(e)
        }

In [3]:
base_path = '/Users/ylee885/Library/CloudStorage/OneDrive-SharedLibraries-GeorgiaInstituteofTechnology/Engle Lab - raw'
txt_path = "/resting_usable_list.txt"

txt_path = os.path.join(base_path, "resting_usable_list.txt")

with open(txt_path, "r") as f:
    file_names = [line.strip() for line in f if line.strip()]

print("Number of usable files:", len(file_names))

Number of usable files: 202


In [4]:
import os

all_results = []
for fname in file_names:

    file_path = os.path.join(base_path, fname)
    subject_id = fname[:5]
    
    if not os.path.exists(file_path):
        print(f"File not found: {file_path}")
        continue
 
    result = process_one_subject(file_path, subject_id)
    all_results.append(result)

NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Setting channel interpolation method to {'eeg': 'spline'}.
Interpolating bad channels.
    Automatic origin fit: head of radius 96.4 mm
Computing interpolation matrix from 63 sensor positions
Interpolating 1 sensors
EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
    Rejecting  epoch based on EEG : ['E32']
    Rejecting  epoch based on EEG : ['E32']
    Rejecting  epoch based on EEG : ['E32']
    Rejecting  epoch based on EEG : ['E32']
    Rejecting  epoch based on EEG : ['E32']
    Rejecting  epoch based on EEG : ['E32']
    Rejecting  epoch based on EEG : ['E32']
    Rejecting  epoch based on EEG : ['E29']
    Rejecting  epoch based on EEG : ['E29', 'E32']
    Rejecting  epoch based on EEG : ['E29']
    Rejecting  epoch based on EEG : ['E47']
    Rejecting  epoch based on EEG : ['E47']
    Rejecting  epoch based on EEG : ['E32']
    Rejecting 

/var/folders/1h/clz66k6s1n3f_89j69yskcm40000gr/T/ipykernel_11654/3767918920.py:71: RuntimeWarning: No bad channels to interpolate. Doing nothing...
  raw_ec.interpolate_bads(reset_bads=True)


    Rejecting  epoch based on EEG : ['E17', 'E61', 'E62']
    Rejecting  epoch based on EEG : ['E1', 'E5', 'E17', 'E61', 'E62', 'E64']
    Rejecting  epoch based on EEG : ['E1', 'E5', 'E17', 'E29', 'E30', 'E61', 'E62']
    Rejecting  epoch based on EEG : ['E17', 'E29', 'E30']
    Rejecting  epoch based on EEG : ['E17']
    Rejecting  epoch based on EEG : ['E10', 'E17']
    Rejecting  epoch based on EEG : ['E17', 'E42', 'E46', 'E47', 'E57']
    Rejecting  epoch based on EEG : ['E17', 'E42', 'E46', 'E47', 'E57']
    Rejecting  epoch based on EEG : ['E2', 'E13', 'E14', 'E19', 'E22', 'E23', 'E26', 'E55', 'E57', 'E58', 'E59', 'E60']
    Rejecting  epoch based on EEG : ['E2', 'E5', 'E12', 'E13', 'E14', 'E18', 'E19', 'E22', 'E23', 'E24', 'E26', 'E29', 'E55', 'E56', 'E57', 'E58', 'E59', 'E60']
    Rejecting  epoch based on EEG : ['E1', 'E2', 'E5', 'E12', 'E13', 'E14', 'E18', 'E19', 'E22', 'E23', 'E24', 'E29', 'E30', 'E55', 'E56', 'E57', 'E58', 'E59', 'E60', 'E61', 'E63']
    Rejecting  epoch b

/var/folders/1h/clz66k6s1n3f_89j69yskcm40000gr/T/ipykernel_11654/3767918920.py:71: RuntimeWarning: No bad channels to interpolate. Doing nothing...
  raw_ec.interpolate_bads(reset_bads=True)


    Rejecting  epoch based on EEG : ['E40']
    Rejecting  epoch based on EEG : ['E40']
    Rejecting  epoch based on EEG : ['E40']
    Rejecting  epoch based on EEG : ['E40']
    Rejecting  epoch based on EEG : ['E40']
    Rejecting  epoch based on EEG : ['E40']
    Rejecting  epoch based on EEG : ['E40']
    Rejecting  epoch based on EEG : ['E40']
    Rejecting  epoch based on EEG : ['E40']
    Rejecting  epoch based on EEG : ['E40']
    Rejecting  epoch based on EEG : ['E40']
    Rejecting  epoch based on EEG : ['E40']
    Rejecting  epoch based on EEG : ['E40']
    Rejecting  epoch based on EEG : ['E40']
    Rejecting  epoch based on EEG : ['E23']
    Rejecting  epoch based on EEG : ['E23']
    Rejecting  epoch based on EEG : ['E23', 'E63']
    Rejecting  epoch based on EEG : ['E23']
    Rejecting  epoch based on EEG : ['E23']
    Rejecting  epoch based on EEG : ['E40']
    Rejecting  epoch based on EEG : ['E40']
    Rejecting  epoch based on EEG : ['E40']
    Rejecting  epoch base

...

NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Setting channel interpolation method to {'eeg': 'spline'}.
Interpolating bad channels.
    Automatic origin fit: head of radius 96.4 mm
Computing interpolation matrix from 62 sensor positions
Interpolating 2 sensors
EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
    Rejecting  epoch based on EEG : ['E52']
    Rejecting  epoch based on EEG : ['E52']
    Rejecting  epoch based on EEG : ['E47']
    Rejecting  epoch based on EEG : ['E47']
    Rejecting  epoch based on EEG : ['E47']
    Rejecting  epoch based on EEG : ['E5']
    Rejecting  epoch based on EEG : ['E5']
    Rejecting  epoch based on EEG : ['E18']
    Rejecting  epoch based on EEG : ['E18']
    Rejecting  epoch based on EEG : ['E4', 'E6', 'E8', 'E11', 'E23', 'E25', 'E26', 'E27', 'E29', 'E32', 'E38', 'E39', 'E40', 'E41', 'E43', 'E49', 'E50', 'E52', 'E55', 'E56', 'E58', 'E59', 'E60', 'E61

/var/folders/1h/clz66k6s1n3f_89j69yskcm40000gr/T/ipykernel_11654/3767918920.py:71: RuntimeWarning: No bad channels to interpolate. Doing nothing...
  raw_ec.interpolate_bads(reset_bads=True)


NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Setting channel interpolation method to {'eeg': 'spline'}.
Interpolating bad channels.
    Automatic origin fit: head of radius 96.4 mm
Computing interpolation matrix from 63 sensor positions
Interpolating 1 sensors
EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
    Rejecting  epoch based on EEG : ['E17', 'E25']
    Rejecting  epoch based on EEG : ['E17', 'E25']
    Rejecting  epoch based on EEG : ['E17']
    Rejecting  epoch based on EEG : ['E17']
    Rejecting  epoch based on EEG : ['E17']
    Rejecting  epoch based on EEG : ['E17']
    Rejecting  epoch based on EEG : ['E62']
    Rejecting  epoch based on EEG : ['E62']
    Rejecting  epoch based on EEG : ['E25']
    Rejecting  epoch based on EEG : ['E25']
    Rejecting  epoch based on EEG : ['E25', 'E27']
    Rejecting  epoch based on EEG : ['E23', 'E25', 'E26', 'E27', 'E28', 'E29']
    Rejec

/var/folders/1h/clz66k6s1n3f_89j69yskcm40000gr/T/ipykernel_11654/3767918920.py:71: RuntimeWarning: No bad channels to interpolate. Doing nothing...
  raw_ec.interpolate_bads(reset_bads=True)


NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Setting channel interpolation method to {'eeg': 'spline'}.
Interpolating bad channels.
    Automatic origin fit: head of radius 96.4 mm
Computing interpolation matrix from 63 sensor positions
Interpolating 1 sensors
EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
    Rejecting  epoch based on EEG : ['E17']
    Rejecting  epoch based on EEG : ['E1']
    Rejecting  epoch based on EEG : ['E1', 'E5']
    Rejecting  epoch based on EEG : ['E1', 'E5']
    Rejecting  epoch based on EEG : ['E1', 'E2', 'E3', 'E5', 'E10', 'E11', 'E12', 'E13', 'E14', 'E15', 'E16', 'E17', 'E18', 'E19', 'E20', 'E21', 'E22', 'E23', 'E24', 'E26', 'E27', 'E28', 'E29', 'E41', 'E46', 'E49', 'E50', 'E51', 'E52', 'E53', 'E54', 'E55', 'E56', 'E57', 'E58', 'E59', 'E60', 'E61', 'E62', 'E63', 'E64']
    Rejecting  epoch based on EEG : ['E1', 'E2', 'E3', 'E7', 'E10', 'E11', 'E12', 'E13',

In [6]:
import pandas as pd
import os

# Convert list of dictionaries to DataFrame
df = pd.DataFrame(all_results)

# Ensure Subject is numeric for proper ascending sort
df["Subject"] = pd.to_numeric(df["Subject"], errors="coerce")

# Sort by Subject in ascending order
df = df.sort_values("Subject")

# Move Subject column to the first position
cols = ["Subject"] + [c for c in df.columns if c != "Subject"]
df = df[cols]

# Save as CSV
save_path = os.path.join('/Users/ylee885/GaTech Dropbox/Yoonsang Lee/2026/Comp Neuro/EEG', "resting_subject_features.csv")
df.to_csv(save_path, index=False)

print("Saved:", save_path)

Saved: /Users/ylee885/GaTech Dropbox/Yoonsang Lee/2026/Comp Neuro/EEG/resting_subject_features.csv


In [ ]:
import pandas as pd
cloud_path = '/Users/ylee885/GaTech Dropbox/Yoonsang Lee/2026/Comp Neuro/EEG'
save_path =  os.path.join(cloud_path, "subject_features.csv")
df = pd.DataFrame(all_results)
df.to_csv(save_path, index=False)

print("Saved to:", save_path)

Number of usable files: 0
